# Deploy Qwen-Image-2512 on SageMaker with SGLang

This notebook demonstrates how to deploy the **Qwen-Image-2512** text-to-image model on Amazon SageMaker using the **Bring Your Own Container (BYOC)** approach with **SGLang** for optimized inference.

## Overview
- **Model**: Qwen-Image-2512 (20B MMDiT)
- **Framework**: SGLang Diffusion
- **Recommended Instance**: ml.g5.12xlarge (4x A10G) or ml.p4de.24xlarge (8x A100)

## SGLang Advantages
- Native Qwen-Image-2512 support in model registry
- **Multi-GPU tensor parallelism** (`--num-gpus`) for distributing model across GPUs
- **CFG parallelism** (`--enable-cfg-parallel`) runs positive/negative prompts in parallel
- 1.2x-5.9x speedup over vanilla Diffusers with CFG parallelism
- OpenAI-compatible `/v1/images/generations` API
- **Automatic fallback** to single-GPU if multi-GPU initialization fails

## Prerequisites
- AWS account with SageMaker access
- IAM role with ECR and SageMaker permissions
- Docker installed locally (for building the container)
- `Dockerfile` and `serve.py` in the same directory as this notebook

---
## Step 1: Setup and Configuration

In [ ]:
# Install required libraries
!pip install -q boto3 sagemaker

In [ ]:
import boto3
import sagemaker
from sagemaker import get_execution_role
import json
import base64
from datetime import datetime

# AWS Configuration
region = boto3.Session().region_name
account_id = boto3.client("sts").get_caller_identity()["Account"]

# SageMaker session and role
sagemaker_session = sagemaker.Session()
role = get_execution_role()  # Or specify your IAM role ARN

# Container configuration
ecr_repo_name = "qwen-image-sglang"
image_tag = "latest"
ecr_image_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{ecr_repo_name}:{image_tag}"

print(f"Region: {region}")
print(f"Account ID: {account_id}")
print(f"ECR Image URI: {ecr_image_uri}")

---
## Step 2: Build and Push Docker Image to ECR

**Option A**: Use the build script (recommended)
```bash
./build-docker.sh
```

**Option B**: Run the cells below to build manually

In [ ]:
# Create ECR repository if it doesn't exist
ecr_client = boto3.client("ecr", region_name=region)

try:
    ecr_client.create_repository(repositoryName=ecr_repo_name)
    print(f"Created ECR repository: {ecr_repo_name}")
except ecr_client.exceptions.RepositoryAlreadyExistsException:
    print(f"ECR repository already exists: {ecr_repo_name}")

In [ ]:
# Authenticate Docker with ECR
!aws ecr get-login-password --region {region} | docker login --username AWS --password-stdin {account_id}.dkr.ecr.{region}.amazonaws.com

In [ ]:
# Build Docker image (this may take 10-15 minutes)
!docker build -t {ecr_repo_name}:{image_tag} .

In [ ]:
# Tag and push image to ECR
!docker tag {ecr_repo_name}:{image_tag} {ecr_image_uri}
!docker push {ecr_image_uri}

print(f"\nImage pushed to: {ecr_image_uri}")

---
## Step 3: Create SageMaker Model

Define the SageMaker model using our custom container.

In [ ]:
from sagemaker.model import Model

# Model name with timestamp for uniqueness
model_name = f"qwen-image-sglang-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

# Create SageMaker Model
model = Model(
    image_uri=ecr_image_uri,
    role=role,
    name=model_name,
    sagemaker_session=sagemaker_session,
    env={
        "MODEL_ID": "Qwen/Qwen-Image-2512",
        "HF_TOKEN": ""  # Add your HuggingFace token if model is gated
    }
)

print(f"Model name: {model_name}")

---
## Step 4: Deploy to SageMaker Endpoint

Deploy the model to a GPU instance:

| Instance | GPUs | VRAM |  Notes |
|----------|------|------|-------|
| ml.g5.12xlarge | 4x A10G | 96GB |  Cost-effective |
| ml.p4de.24xlarge | 8x A100 | 640GB |  Maximum performance |

**Features enabled:**
- Multi-GPU tensor parallelism for distributing the 20B model
- CFG parallelism for 1.2x-5.9x speedup
- Automatic fallback to single-GPU if multi-GPU fails

In [ ]:
# Endpoint configuration
endpoint_name = f"qwen-image-sglang-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

# Choose your instance type
instance_type = "ml.g5.12xlarge"  # Cost-effective option
# instance_type = "ml.p4de.24xlarge"  # Maximum performance option

print(f"Deploying to endpoint: {endpoint_name}")
print(f"Instance type: {instance_type}")
print("\nThis will take approximately 15-30 minutes (includes model download and multi-GPU initialization)...")

# Deploy the model with extended timeout for multi-GPU initialization
predictor = model.deploy(
    initial_instance_count=1,
    instance_type=instance_type,
    endpoint_name=endpoint_name,
    container_startup_health_check_timeout=1800,  # 30 min for model loading
)

print(f"\nEndpoint deployed: {endpoint_name}")

---
## Step 5: Test the Endpoint

Send a text prompt and receive a generated image.

**Qwen-Image recommended parameters:**
- `num_inference_steps`: 50
- `guidance_scale`: 4.0

In [ ]:
import json
import base64
import random
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt

# Create SageMaker runtime client
runtime_client = boto3.client("sagemaker-runtime", region_name=region)

In [ ]:
%%time
# Test prompt
prompt = """
Realistic still life photography style: A single, fresh apple resting on a clean, soft-textured surface. 
The apple is slightly off-center, softly backlit to highlight its natural gloss and subtle color gradients.
Fine details such as small blemishes, dew drops, and light highlights enhance its lifelike appearance. 
Shallow depth of field, warm tones, hyper-detailed 8K resolution, studio lighting, photorealistic render.
"""

# Use random seed for variety
seed = random.randint(1, 2**32 - 1)

payload = {
    "prompt": prompt,
    "negative_prompt": "blurry, low quality, distorted",
    "width": 1024,
    "height": 1024,
    "num_inference_steps": 50,  # Qwen-Image recommended
    "guidance_scale": 4.0,       # Qwen-Image recommended (maps to true_cfg_scale)
    "seed": seed
}

print(f"Generating image...")
print(f"Seed: {seed}")

# Invoke endpoint
response = runtime_client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Body=json.dumps(payload)
)

# Parse response
result = json.loads(response["Body"].read().decode())
print(f"Generation complete! Seed used: {result['seed']}")

In [ ]:
# Decode and display the generated image
image_data = base64.b64decode(result["image"])
image = Image.open(BytesIO(image_data))

plt.figure(figsize=(10, 10))
plt.imshow(image)
plt.axis("off")
plt.title(f"Generated Image (seed: {result['seed']})")
plt.show()

# Optionally save the image
# image.save("generated_image.png")
# print("Image saved to: generated_image.png")

---
## Step 6: Cleanup Resources

Delete the endpoint and associated resources when done to avoid ongoing charges.

In [ ]:
# Uncomment the following lines to delete resources

# Delete endpoint
# predictor.delete_endpoint()
# print(f"Deleted endpoint: {endpoint_name}")

# Delete model
# predictor.delete_model()
# print(f"Deleted model: {model_name}")

print("To cleanup, uncomment and run the delete commands above.")

---
## Summary

This notebook demonstrated:
1. Building a custom Docker container with SGLang Diffusion
2. Pushing the container to Amazon ECR
3. Deploying Qwen-Image-2512 to SageMaker
4. Invoking the endpoint for text-to-image generation

**Key Resources:**
- [Qwen-Image GitHub](https://github.com/QwenLM/Qwen-Image)
- [SGLang Diffusion Blog](https://lmsys.org/blog/2025-11-07-sglang-diffusion/)
- [SGLang Documentation](https://docs.sglang.io)
- [SageMaker BYOC Guide](https://docs.aws.amazon.com/sagemaker/latest/dg/docker-containers.html)